# 04 — Feature Engineering

1. Correlation matrix — check multicollinearity
2. Normalization — z-score vs percentile rank
3. Feature importance and selection

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')

In [ ]:
features = pd.read_parquet('../../data/processed/features.parquet')
print(f'Shape: {features.shape}')
features.head()

In [ ]:
# Correlation analysis
from analytics.scoring.correlation import analyze_feature_correlations

corr, high_corr = analyze_feature_correlations(features)

if high_corr:
    print('\nAction items for multicollinear pairs:')
    from analytics.scoring.correlation import recommend_feature_actions
    recommend_feature_actions(high_corr)

In [ ]:
# Normalization
from analytics.scoring.normalize import normalize_features

normalized = normalize_features(features)

# Compare before/after distributions
numeric_cols = features.select_dtypes(include='number').columns[:6]
fig, axes = plt.subplots(2, len(numeric_cols), figsize=(18, 6))
for i, col in enumerate(numeric_cols):
    features[col].dropna().hist(bins=15, ax=axes[0, i], color='#ef4444', alpha=0.7)
    axes[0, i].set_title(f'{col}\n(raw)', fontsize=8)
    normalized[col].dropna().hist(bins=15, ax=axes[1, i], color='#22c55e', alpha=0.7)
    axes[1, i].set_title(f'{col}\n(normalized)', fontsize=8)
plt.suptitle('Raw vs Normalized Feature Distributions', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Skewness report
print('Feature skewness (|skew| > 2 → percentile rank, else z-score):')
for col in features.select_dtypes(include='number').columns:
    skew = features[col].skew()
    method = 'percentile' if abs(skew) > 2 else 'z-score'
    flag = ' ⚠️' if abs(skew) > 2 else ''
    print(f'  {col:<35s} skew={skew:+.2f}  → {method}{flag}')